In [ ]:
import pandas as pd
import numpy as np
import mysql.connector
from datetime import datetime

# =========================
# DB CONNECTION
# =========================
def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="12345678",
        database="marketing_campaign"
    )

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("E:/AI-ML/marketing_campaign/data/marketing_campaign_data.csv")
dict_df = pd.read_csv("E:/AI-ML/marketing_campaign/data/marketing_data_dictionary.csv")

df.columns = df.columns.str.strip()
dict_df['Field'] = dict_df['Field'].str.strip()



print("Data Loaded Successfully")

# =========================
# BASIC CLEANING
# =========================
# Drop columns with >30% missing
df = df[df.columns[df.isna().mean() < 0.3]]

# Fill missing values
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# Convert Date
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], errors='coerce')

print("Missing Values Handled")

# =========================
# FEATURE ENGINEERING
# =========================
CURRENT_YEAR = datetime.now().year

df['Age'] = CURRENT_YEAR - df['Year_Birth']
df['TotalChildren'] = df['Kidhome'] + df['Teenhome']

df['TotalSpend'] = (
    df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] +
    df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']
)

df['TotalPurchases'] = (
    df['NumWebPurchases'] + df['NumCatalogPurchases'] +
    df['NumStorePurchases'] + df['NumDealsPurchases']
)
numeric_cols = [
    'Income', 'Kidhome', 'Teenhome', 'MntWines', 'MntFruits',
    'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds',
    'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumDealsPurchases',
    'NumWebVisitsMonth', 'Recency', 'Response'
]

# Fill NaNs with 0
df[numeric_cols] = df[numeric_cols].fillna(0)

df['Year_Birth'] = df['Year_Birth'].fillna(df['Year_Birth'].median())
df['Age'] = pd.Timestamp.now().year - df['Year_Birth']

# Keep only columns present in dictionary
df = df[dict_df['Field']]
print("Feature Engineering Completed")

# =========================
# OUTLIER REMOVAL (IQR)
# =========================
def remove_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[col] >= lower) & (df[col] <= upper)]

for col in ['Income', 'TotalSpend', 'Age']:
    df = remove_outliers(df, col)

print("Outliers Removed")


print("Segmentation Completed")

# =========================
# CREATE TABLE IN MYSQL
# =========================
def map_dtype(field_name):
    field = field_name.lower()

    if field == 'id':
        return 'INT PRIMARY KEY'
    elif 'year' in field:
        return 'INT'
    elif field == 'income':
        return 'DECIMAL(12,2)'
    elif field.startswith('mnt'):
        return 'DECIMAL(10,2)'
    elif field.startswith('num'):
        return 'INT'
    elif field in ['kidhome', 'teenhome', 'recency', 'response', 'complain',
                   'acceptedcmp1', 'acceptedcmp2', 'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5']:
        return 'INT'
    elif field == 'dt_customer':
        return 'DATE'
    else:
        return 'VARCHAR(100)'

table_name = "customer_marketing_raw"

sql_lines = [f"CREATE TABLE IF NOT EXISTS {table_name} ("]
for i, field in enumerate(dict_df['Field']):
    col = field.strip()
    dtype = map_dtype(col)
    comma = "," if i < len(dict_df['Field']) - 1 else ""
    sql_lines.append(f"  {col} {dtype}{comma}")
sql_lines.append(");")

create_table_sql = "\n".join(sql_lines)

conn = get_connection()
cursor = conn.cursor()

print("Table Created / Verified")

# =========================
# TRUNCATE TABLE
# =========================
cursor.execute(f"DROP TABLE {table_name}")
conn.commit()
print("Table Droped")

cursor.execute(create_table_sql)
conn.commit()
# =========================
# INSERT DATA
# =========================
df = df.replace({np.nan: None})
columns = list(df.columns)
placeholders = ", ".join(["%s"] * len(columns))
col_names = ", ".join(columns)

insert_sql = f"""
INSERT INTO {table_name} ({col_names})
VALUES ({placeholders})
"""

data_tuples = [tuple(row) for row in df.to_numpy()]

cursor.executemany(insert_sql, data_tuples)
conn.commit()

print(f"{cursor.rowcount} Rows Inserted Successfully")

print("DATA PIPELINE COMPLETED SUCCESSFULLY")
cursor = conn.cursor(dictionary=True)

query = "SELECT * FROM customer_marketing_raw"
cursor.execute(query)

rows = cursor.fetchall()
df = pd.DataFrame(rows)

print(df.head())


Data Loaded Successfully
Missing Values Handled
Feature Engineering Completed
Outliers Removed
Segmentation Completed
Table Created / Verified
Table Truncated
55281 Rows Inserted Successfully
DATA PIPELINE COMPLETED SUCCESSFULLY
    ID  Year_Birth   Education Marital_Status     Income  Kidhome  Teenhome  \
0   36        1961      Master       Divorced   14589.60        1         1   
1  276        1967  Graduation           YOLO   77367.20        0         1   
2  541        1973      Master       Together   15654.80        1         1   
3  624        1977  Graduation       Together  114325.30        0         0   
4  830        1969  Graduation       Together    1730.00        1         0   

  Dt_Customer  Recency MntWines  ... AcceptedCmp5 AcceptedCmp1 AcceptedCmp2  \
0  2014-02-04       14   232.00  ...            0            0            0   
1  2013-02-28       80   145.00  ...            0            0            0   
2  2013-08-06       92     0.00  ...            0          